# EN4554 Assignment 2: CNNs
**Name:** YOUR NAME
**Index:** YOUR INDEX NUMBER
**GitHub:** https://github.com/YOUR_USERNAME/en4554-a2

Experiments were run as scripts on a GPU server (NVIDIA RTX 6000 Ada); this notebook shows the key code and the saved results. The use of AI tools is acknowledged at the end.

In [ ]:
import os, json
import pandas as pd
from IPython.display import Image, display
pd.set_option('display.precision', 4)
R = 'results'

## Setup
45k/5k split of the CIFAR-10 training set with a fixed seed, per-channel normalisation with the 45k training statistics, seeds fixed. Test set only used in Task 4.

```python
def load_cifar10(args, load_test=False, device=DEVICE):
    """45k train / 5k val split of the CIFAR-10 training set, normalised per
    channel with the statistics of the 45k training images only."""
    if args.fake:
        rng = np.random.RandomState(123)
        xtr = rng.randint(0, 256, (50000, 32, 32, 3), dtype=np.uint8)
        ytr = rng.randint(0, 10, 50000)
        xte = rng.randint(0, 256, (10000, 32, 32, 3), dtype=np.uint8)
        yte = rng.randint(0, 10, 10000)
    else:
        from torchvision.datasets import CIFAR10
        tr = CIFAR10(args.data, train=True, download=True)
        xtr, ytr = tr.data, np.array(tr.targets)
        if load_test:                       # test set is touched ONLY in Task 4
            te = CIFAR10(args.data, train=False, download=True)
            xte, yte = te.data, np.array(te.targets)

    perm = np.random.RandomState(SPLIT_SEED).permutation(50000)
    tr_idx, va_idx = perm[:45000], perm[45000:]

    def to_tensor(a):                       # uint8 NHWC -> float NCHW in [0,1]
        return torch.from_numpy(a).permute(0, 3, 1, 2).float().div_(255)

    x = to_tensor(xtr)
    mean = x[tr_idx].mean(dim=(0, 2, 3))    # training statistics only
    std = x[tr_idx].std(dim=(0, 2, 3))

    def norm(t):
        return ((t - mean.view(1, 3, 1, 1)) / std.view(1, 3, 1, 1)).to(device)

    y = torch.from_numpy(ytr).long()
    d = dict(xtr=norm(x[tr_idx]), ytr=y[tr_idx].to(device),
             xva=norm(x[va_idx]), yva=y[va_idx].to(device),
             mean=mean.tolist(), std=std.tolist())
    if load_test:
        d["xte"] = norm(to_tensor(xte))
        d["yte"] = torch.from_numpy(yte).long().to(device)
    if args.quick:                          # shrink everything for a smoke test
        d["xtr"], d["ytr"] = d["xtr"][:2000], d["ytr"][:2000]
        d["xva"], d["yva"] = d["xva"][:1000], d["yva"][:1000]
        if load_test:
            d["xte"], d["yte"] = d["xte"][:1000], d["yte"][:1000]
    return d
```

## Task 1: Backpropagation
### (a) Layers with explicit forward/backward

```python
class Linear:
    def __init__(self, din, dout, init="kaiming", dtype=torch.float32, device=DEVICE):
        std = {"kaiming": math.sqrt(2.0 / din),          # He, for ReLU
               "xavier": math.sqrt(2.0 / (din + dout)),  # Glorot, for sigmoid/tanh
               "lecun": math.sqrt(1.0 / din)}[init]
        self.W = torch.randn(din, dout, dtype=dtype, device=device) * std
        self.b = torch.zeros(dout, dtype=dtype, device=device)
        self.dW, self.db = torch.zeros_like(self.W), torch.zeros_like(self.b)

    def forward(self, x):                 # y = x W + b
        self.x = x
        return x @ self.W + self.b

    def backward(self, g):                # g = dL/dy  (N x dout)
        self.dW = self.x.T @ g            # dL/dW = x^T g
        self.db = g.sum(0)                # dL/db = sum over batch
        return g @ self.W.T               # dL/dx = g W^T

    def params(self):
        return [self.W, self.b]

    def grads(self):
        return [self.dW, self.db]


class ReLU:
    def forward(self, x):
        self.mask = (x > 0).to(x.dtype)
        return x * self.mask

    def backward(self, g):                # derivative is 1 where x > 0
        return g * self.mask

    def params(self):
        return []

    def grads(self):
        return []


class Sigmoid:
    def forward(self, x):
        self.s = 1.0 / (1.0 + torch.exp(-x))
        return self.s

    def backward(self, g):                # sigma' = sigma (1 - sigma) <= 1/4
        return g * self.s * (1 - self.s)

    def params(self):
        return []

    def grads(self):
        return []


class SoftmaxCrossEntropy:
    def forward(self, logits, y):
        z = logits - logits.max(1, keepdim=True).values   # numerical stability
        logp = z - z.exp().sum(1, keepdim=True).log()       # log-softmax
        self.p, self.y = logp.exp(), y
        return -logp[torch.arange(len(y)), y].mean()

    def backward(self):                   # dL/dz = (softmax - onehot) / N
        g = self.p.clone()
        g[torch.arange(len(self.y)), self.y] -= 1
        return g / len(self.y)


class MLP:
    def __init__(self, sizes, act="relu", init=None, dtype=torch.float32, device=DEVICE):
        init = init or ("kaiming" if act == "relu" else "xavier")
        self.layers = []
        for i, (a, b) in enumerate(zip(sizes[:-1], sizes[1:])):
            self.layers.append(Linear(a, b, init, dtype, device))
            if i < len(sizes) - 2:
                self.layers.append(ReLU() if act == "relu" else Sigmoid())
        self.criterion = SoftmaxCrossEntropy()

    def forward(self, x):
        for layer in self.layers:
            x = layer.forward(x)
        return x

    def loss(self, x, y):
        return self.criterion.forward(self.forward(x), y)

    def backward(self):
        g = self.criterion.backward()
        for layer in reversed(self.layers):
            g = layer.backward(g)

    def linears(self):
        return [l for l in self.layers if isinstance(l, Linear)]

    def params(self):
        return [p for l in self.layers for p in l.params()]

    def grads(self):
        return [g for l in self.layers for g in l.grads()]

    def sgd_step(self, lr):
        for p, g in zip(self.params(), self.grads()):
            p.sub_(lr * g)
```

### (b) Gradient check

```python
def autograd_grads(mlp, x, y):
    """Same network rebuilt with torch ops + autograd, sharing the weights."""
    ps = [p.detach().clone().requires_grad_(True) for p in mlp.params()]
    h, k = x, 0
    for layer in mlp.layers:
        if isinstance(layer, Linear):
            h = h @ ps[k] + ps[k + 1]
            k += 2
        elif isinstance(layer, ReLU):
            h = torch.relu(h)
        else:
            h = torch.sigmoid(h)
    loss = F.cross_entropy(h, y)
    loss.backward()
    return [p.grad for p in ps]


def finite_diff(mlp, x, y, p, idx, eps=1e-6):
    """Centred difference (L(p+eps) - L(p-eps)) / 2eps at flat indices idx."""
    flat, out = p.view(-1), []
    for i in idx:
        old = flat[i].item()
        flat[i] = old + eps
        lp = mlp.loss(x, y).item()
        flat[i] = old - eps
        lm = mlp.loss(x, y).item()
        flat[i] = old
        out.append((lp - lm) / (2 * eps))
    return torch.tensor(out, dtype=p.dtype, device=p.device)


def rel_err(a, b):
    return ((a - b).norm() / (a.norm() + b.norm() + 1e-30)).item()
```

In [ ]:
display(pd.read_csv(os.path.join(R, 'task1/gradcheck.csv')))

**Interpretation.**

- **vs autograd:** the relative error is exactly 0 for every ReLU-network parameter and at most 9e-17 for the sigmoid network. Float64 machine epsilon is about 2.2e-16, so the manual gradients are identical to autograd up to rounding, and `allclose` is `True` for all 18 parameter tensors. The sigmoid case is not exactly 0 only because `σ(z)(1−σ(z))` is computed in a slightly different order from PyTorch.
- **vs finite differences:** the error is 3e-10 to 2e-8. This is larger, but expected, because the centred difference is itself only an approximation:
  - truncation error O(ε²) from the Taylor expansion, and
  - round-off error O(machine ε / ε) from subtracting two nearly equal losses.
  - With ε ≈ 1e-6 both terms are around 1e-10 to 1e-9, which is what we see. Sigmoid has slightly larger FD error (up to 1.6e-8) because its curvature makes the O(ε²) term bigger than for piecewise-linear ReLU.
- The check covers every entry of the tiny networks and 40 random entries per tensor of the full CIFAR-sized network (3072→128→64→10), so the shapes used in training are tested too.

Overall, the forward/backward passes of Linear, ReLU, Sigmoid and softmax cross-entropy are correct.

### (c) SGD training on flattened CIFAR-10

In [ ]:
display(Image(filename=os.path.join(R, 'task1/mlp_sgd_training.png')))

In [ ]:
print(json.load(open(os.path.join(R,'task1/mlp_sgd_summary.json'))))

**Interpretation.**

- The MLP (3072→512→256→10, SGD lr 0.02, 30 epochs) reaches about **50% validation accuracy by epoch 5** and then stays there: the best is 51.5% (epoch 29) and the final value is 50.3%.
- The **training loss keeps decreasing** all the way (1.75 → 0.20), but the **validation loss is lowest at epoch 6 (1.45) and then climbs to 2.15**. This growing train/val gap is overfitting: the network is memorising the 45k training images rather than learning features that transfer. Accuracy stays flat while the validation loss rises because the model becomes over-confident on the images it gets wrong.
- An MLP on raw pixels is limited for images:
  - Flattening throws away the 2-D layout, so the network has no notion of neighbouring pixels.
  - There is no weight sharing, so a pattern learnt at one position has to be learnt again at every other position (no translation invariance).
  - This costs ~1.7M parameters in the first layer alone, which makes memorisation easy.

Early stopping around epoch 6 would give the best model, and it still tops out at about 50%. The CNNs in Task 2(c) reach ~75% with far fewer parameters.

### (d) Per-layer gradient norms in a 10-layer MLP

In [ ]:
display(Image(filename=os.path.join(R, 'task1/grad_norms.png')))

**Interpretation.**

- **Sigmoid (Xavier):** the gradient norm falls from 1.65 at layer 10 to 1.2e-5 at layer 1, about **1.4×10⁵ times smaller**, and roughly 4× smaller per layer (the line is straight on the log scale).
  - The sigmoid derivative σ'(z) = σ(1−σ) is at most 1/4. Each layer the backward signal is multiplied by σ' ≤ 1/4 (times the weights), so it shrinks geometrically: 0.25⁹ ≈ 4×10⁻⁶ over 9 layers, the same order as observed.
  - The first layers would therefore barely learn. This is the **vanishing gradient** problem.
- **ReLU (Kaiming):** the norm stays between about 2 and 5.5 across all 10 layers. ReLU's derivative is exactly 1 for active units (0 otherwise), and Kaiming init (variance 2/fan_in) compensates for the half of the units that are off, so the variance of the signal is kept constant from layer to layer in both directions.
- **Mismatched initialisations** separate the effect of the activation from the effect of the init:
  - **Sigmoid with Kaiming init** still vanishes at the same rate (1.77 → 1.3e-4). Larger weights cannot fix it, because the ≤ 1/4 factor comes from the activation itself.
  - **ReLU with Xavier init** is still flat across layers, but every layer's gradient is about 50× smaller (~0.06). Xavier (variance 1/fan_in) does not account for ReLU zeroing half the units, so the activations shrink in the forward pass and every gradient is scaled down. Its shape stays flat because ReLU does not multiply by a factor < 1 per layer.

So the activation function decides whether gradients vanish *with depth*, and the initialisation sets their overall *scale*.

## Task 2: Optimizers
### (a) Custom SGD-momentum and Adam

```python
class MySGD(torch.optim.Optimizer):
    """v_t = mu * v_{t-1} + g_t   (v_1 = g_1);   theta <- theta - lr * v_t
    (with nesterov: theta <- theta - lr * (g_t + mu * v_t)).
    weight_decay adds wd * theta to the gradient (L2 regularisation)."""
    def __init__(self, params, lr=0.01, momentum=0.0, weight_decay=0.0, nesterov=False):
        if lr <= 0:
            raise ValueError("lr must be positive")
        super().__init__(params, dict(lr=lr, momentum=momentum,
                                      weight_decay=weight_decay, nesterov=nesterov))

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        for group in self.param_groups:
            lr, mu = group["lr"], group["momentum"]
            wd, nesterov = group["weight_decay"], group["nesterov"]
            for p in group["params"]:
                if p.grad is None:
                    continue
                g = p.grad
                if wd != 0:
                    g = g.add(p, alpha=wd)
                if mu != 0:
                    state = self.state[p]
                    buf = state.get("momentum_buffer")
                    if buf is None:
                        buf = g.clone()
                        state["momentum_buffer"] = buf
                    else:
                        buf.mul_(mu).add_(g)
                    g = g.add(buf, alpha=mu) if nesterov else buf
                p.add_(g, alpha=-lr)
        return loss


class MyAdam(torch.optim.Optimizer):
    """m_t = b1 m + (1-b1) g,   v_t = b2 v + (1-b2) g^2
    m_hat = m_t / (1-b1^t),     v_hat = v_t / (1-b2^t)      (bias correction)
    theta <- theta - lr * m_hat / (sqrt(v_hat) + eps)
    Implemented in the algebraically identical form PyTorch uses:
    theta <- theta - (lr/(1-b1^t)) * m_t / (sqrt(v_t)/sqrt(1-b2^t) + eps)."""
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.0):
        super().__init__(params, dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay))

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        for group in self.param_groups:
            lr, (b1, b2) = group["lr"], group["betas"]
            eps, wd = group["eps"], group["weight_decay"]
            for p in group["params"]:
                if p.grad is None:
                    continue
                g = p.grad
                if wd != 0:
                    g = g.add(p, alpha=wd)
                state = self.state[p]
                if len(state) == 0:
                    state["step"] = 0
                    state["m"] = torch.zeros_like(p)
                    state["v"] = torch.zeros_like(p)
                state["step"] += 1
                t, m, v = state["step"], state["m"], state["v"]
                m.mul_(b1).add_(g, alpha=1 - b1)
                v.mul_(b2).addcmul_(g, g, value=1 - b2)
                bc1 = 1 - b1 ** t
                bc2 = 1 - b2 ** t
                denom = (v.sqrt() / math.sqrt(bc2)).add_(eps)
                p.addcdiv_(m, denom, value=-lr / bc1)
        return loss
```

In [ ]:
display(pd.read_csv(os.path.join(R, 'task2/optimizer_match.csv')))

In [ ]:
display(Image(filename=os.path.join(R, 'task2/optimizer_match.png')))

**Interpretation.**

- **SGD (plain, momentum, momentum + weight decay, Nesterov):** the maximum parameter difference from `torch.optim.SGD` after 200 steps is **exactly 0**. The implementation performs the same floating-point operations in the same order as PyTorch, so the results are bit-for-bit identical.
- **Adam (default and with betas = (0.8, 0.99) + weight decay):** the difference is **1.1e-16**, i.e. one unit of float64 rounding. The bias-corrected update is mathematically the same as PyTorch's; only the order of a few operations differs.
- `allclose` is `True` in all six configurations, and the plot shows the two parameter trajectories lying on top of each other.

Both optimizers, including momentum, Nesterov, weight decay and Adam's bias correction, are implemented correctly.

### (b) Trajectories on f(x,y) = (x^2 + 50 y^2)/2

In [ ]:
display(Image(filename=os.path.join(R, 'task2/quadratic_trajectories.png')))

**Interpretation.**

The Hessian is diag(1, 50), so the condition number is **50**: the y-direction is 50× steeper than the x-direction.

- **SGD (lr = 0.035)**
  - Gradient descent on a quadratic is stable only when lr < 2/λ_max = 2/50 = 0.04, so the steep y-direction caps the learning rate.
  - In y the per-step factor is 1 − 0.035·50 = −0.75: the sign flips every step, which is the **zig-zag** seen in the first ~15 steps, before y converges.
  - In x the factor is 1 − 0.035 = 0.965, so it only moves 3.5% closer per step and **crawls**: after 100 steps x = −10·0.965¹⁰⁰ ≈ −0.28 and f = 0.040, a straight line on the log plot.
- **SGD + momentum (lr = 0.01, μ = 0.9)**
  - Momentum accumulates the consistent gradient along x, so the effective step along the valley is about lr/(1−μ) = 0.1, much larger than plain SGD can use.
  - The alternating y-gradients cancel out in the velocity, which damps the zig-zag.
  - It overshoots and spirals around the minimum (x reaches about +1.5), so the loss curve is bumpy, but it ends **closest to the minimum (f = 5.6e-4)**.
- **Adam (lr = 0.3)**
  - Adam divides each coordinate's step by the running RMS of its gradient, so x and y get steps of similar size regardless of curvature. It therefore moves **diagonally** at first and follows a smooth path, without SGD's zig-zag.
  - Near the minimum the normalised step does not shrink with the gradient, so it oscillates on a scale set by lr and ends at f = 1.4e-3.

Momentum fixes ill-conditioning by building speed along the shallow direction, Adam by rescaling each coordinate. Both reach f ≈ 1e-3 where plain SGD is still at 4e-2.

### (c) Small CNN: 3 optimizers x 3 learning rates

```python
class SmallCNN(nn.Module):
    """Task 2(c): small CNN, no batch norm (so the optimiser matters more)."""
    def __init__(self, num_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),    # 16x16
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),   # 8x8
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),  # 4x4
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(inplace=True),
            nn.Linear(256, num_classes))

    def forward(self, x):
        return self.net(x)


def conv_bn_relu(cin, cout):
    return nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1, bias=False),
                         nn.BatchNorm2d(cout), nn.ReLU(inplace=True))


class SmallBNCNN(nn.Module):
    """Task 3(a): small CNN with batch norm and optional dropout in the head."""
    def __init__(self, dropout=0.0, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            conv_bn_relu(3, 32), conv_bn_relu(32, 32), nn.MaxPool2d(2),
            conv_bn_relu(32, 64), conv_bn_relu(64, 64), nn.MaxPool2d(2),
            conv_bn_relu(64, 128), nn.MaxPool2d(2))
        self.head = nn.Sequential(
            nn.Flatten(), nn.Dropout(dropout),
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(256, num_classes))

    def forward(self, x):
        return self.head(self.features(x))
```

In [ ]:
display(pd.read_csv(os.path.join(R, 'task2/lr_sweep_table.csv')))

In [ ]:
display(Image(filename=os.path.join(R, 'task2/lr_sweep.png')))

**Interpretation.**

Best final validation accuracy after 15 epochs: **SGD 75.0% (lr 0.05)**, **Adam 74.8% (lr 1e-3, best 75.7%)**, **SGD + momentum 74.3% (lr 0.01, best 75.4%)**. At their best learning rate all three end within about 1 point of each other.

**Sensitivity to the learning rate**
- **Adam is the most sensitive.** At lr = 1e-2 it never learns: from the first epoch the loss is stuck at 2.303 = ln 10 and accuracy is 10% (chance). The large early steps most likely pushed the ReLU units into the dead region, so no gradient flows. It is not flagged as diverged because the loss never becomes NaN. At 1e-4 it is too slow (66.6%).
- **SGD + momentum:** with μ = 0.9 the effective step is about 10 × lr. At 0.05 (effective 0.5) the training loss stops falling after epoch 9 and rises again, and validation accuracy drops from 72.9% to 67.7%. At 0.001 it is too slow (62.7%).
- **Plain SGD is the least sensitive in this grid:** 0.05 and 0.2 differ by only 1.6 points, and only 0.01 is clearly too slow (61.9%).

**Speed at the best learning rate.** After epoch 1 the training loss is 1.42 for Adam, 1.76 for momentum and 1.94 for SGD, so Adam converges fastest, then momentum, then SGD.

**Generalisation.** Faster optimisation did not mean better generalisation. Adam and momentum reach ~97% train accuracy, and their validation loss is lowest around epoch 6 to 7 (0.72 / 0.76) and then rises to 1.31 / 1.15, i.e. they overfit. Plain SGD at 0.05 is at 88% train accuracy with its validation loss still falling (0.81), giving the smallest train/val gap.

Overall, each optimizer only works within about one order of magnitude of learning rate, so tuning the learning rate matters more than the choice of optimizer.

## Task 3: Regularization and residual connections
### (a) Regularization on 5k images

```python
def random_flip(x):
    """Horizontal flip of each image with probability 0.5."""
    mask = torch.rand(x.shape[0], device=x.device) < 0.5
    return torch.where(mask[:, None, None, None], x.flip(3), x)


def random_crop(x, pad=4):
    """Zero-pad by `pad` pixels and take a random 32x32 crop per image."""
    B, C, H, W = x.shape
    xp = F.pad(x, (pad, pad, pad, pad))
    i = torch.randint(0, 2 * pad + 1, (B,), device=x.device)
    j = torch.randint(0, 2 * pad + 1, (B,), device=x.device)
    rows = i[:, None] + torch.arange(H, device=x.device)[None]      # B x H
    cols = j[:, None] + torch.arange(W, device=x.device)[None]      # B x W
    b = torch.arange(B, device=x.device)[:, None, None, None]
    c = torch.arange(C, device=x.device)[None, :, None, None]
    return xp[b, c, rows[:, None, :, None], cols[:, None, None, :]]


def cutout(x, size=16):
    """Cutout (DeVries & Taylor 2017): zero a random size x size square per
    image. The centre can be near the border, so the square may be clipped."""
    B, C, H, W = x.shape
    cy = torch.randint(0, H, (B, 1, 1), device=x.device)
    cx = torch.randint(0, W, (B, 1, 1), device=x.device)
    ys = torch.arange(H, device=x.device)[None, :, None]
    xs = torch.arange(W, device=x.device)[None, None, :]
    h = size // 2
    inside = (ys >= cy - h) & (ys < cy + h) & (xs >= cx - h) & (xs < cx + h)
    return x * (~inside)[:, None].to(x.dtype)


def mixup(x, y, alpha):
    """Mixup (Zhang et al. 2018). Returns mixed inputs, both label sets and
    lambda; the loss is lam*CE(out, y_a) + (1-lam)*CE(out, y_b)."""
    lam = float(np.random.beta(alpha, alpha))
    perm = torch.randperm(x.shape[0], device=x.device)
    return lam * x + (1 - lam) * x[perm], y, y[perm], lam
```

```python
CONFIGS = {
    "baseline":     dict(),
    "weight_decay": dict(wd=5e-4),
    "dropout":      dict(dropout=0.5),
    "flip+crop":    dict(aug=["flip", "crop"]),
    "cutout":       dict(aug=["cutout"]),
    "mixup":        dict(mixup=0.4),
    "all":          dict(wd=5e-4, dropout=0.5, aug=["flip", "crop", "cutout"], mixup=0.4),
}
```

In [ ]:
display(pd.read_csv(os.path.join(R, 'task3a/table.csv')))

In [ ]:
display(Image(filename=os.path.join(R, 'task3a/curves_grid.png')))

In [ ]:
display(Image(filename=os.path.join(R, 'task3a/val_curves.png')))

**Interpretation.**

With only 5k training images the **baseline memorises the training set**: clean train accuracy is 100% from about epoch 29, but validation accuracy is only **66.9%**, a gap of 0.33.

| Method | Train acc (clean) | Val acc | Gap | Change in val acc |
|---|---|---|---|---|
| Baseline | 100% | 66.9% | 0.33 | – |
| Flip + crop | 99.2% | **76.6%** | 0.23 | **+9.7** |
| Dropout 0.5 | 99.9% | 71.2% | 0.29 | +4.4 |
| All combined | 76.9% | 70.0% | **0.07** | +3.2 |
| Cutout | 100% | 68.8% | 0.31 | +1.9 |
| Weight decay | 100% | 67.3% | 0.33 | +0.5 |
| Mixup | 100% | 65.3% | 0.35 | −1.5 |

- **Flip + crop helps most**: it creates realistic new images (a mirrored or shifted car is still a car), so the model sees more varied data. It shrinks the gap and gives the largest validation gain.
- **Dropout** slows memorisation (train accuracy rises much more gradually) and adds 4.4 points.
- **Cutout and weight decay barely help** here. Cutout alone still lets the network reach 100% clean train accuracy. With batch norm, weight decay mostly changes the effective learning rate of the scale-invariant convolution weights rather than limiting capacity.
- **Mixup**: the accuracy measured on the mixed training batches is low (~46%) because the targets are soft blends of two labels, which is why train accuracy here is measured on clean images. On clean images it still reaches 100%, so mixup did not stop memorisation, and validation accuracy is slightly lower than the baseline.
- **All combined** gives by far the smallest gap (0.07), but only because train accuracy drops to 77%: the model now **underfits**. Its validation accuracy is 6.6 points below flip + crop alone.

The effects are **not additive**: stacking every regulariser was worse than the best single one. A small gap is not the goal by itself; the aim is the highest validation accuracy, which needs the right amount of regularisation.

### (b) Plain vs residual networks (depth 8, 20, 56)

```python
class ShortcutA(nn.Module):
    """Parameter-free shortcut (option A of He et al. 2016): subsample by the
    stride and zero-pad the extra channels. Using it means the plain and the
    residual networks have exactly the same parameters."""
    def __init__(self, cin, cout, stride):
        super().__init__()
        self.stride, self.extra = stride, cout - cin

    def forward(self, x):
        if self.stride > 1:
            x = x[:, :, ::self.stride, ::self.stride]
        if self.extra > 0:
            x = F.pad(x, (0, 0, 0, 0, self.extra // 2, self.extra - self.extra // 2))
        return x


class BasicBlock(nn.Module):
    def __init__(self, cin, cout, stride, residual):
        super().__init__()
        self.conv1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(cout)
        self.conv2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(cout)
        self.residual = residual
        self.shortcut = ShortcutA(cin, cout, stride)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.residual:                 # <-- the ONLY difference plain/residual
            out = out + self.shortcut(x)
        return F.relu(out)


class CifarResNet(nn.Module):
    """Depth 6n+2: conv -> 3 stages of n blocks (16/32/64 ch) -> GAP -> FC."""
    def __init__(self, depth, residual=True, widths=(16, 32, 64), num_classes=10):
        super().__init__()
        assert (depth - 2) % 6 == 0, "depth must be 6n+2"
        n = (depth - 2) // 6
        self.conv = nn.Conv2d(3, widths[0], 3, 1, 1, bias=False)
        self.bn = nn.BatchNorm2d(widths[0])
        blocks, cin = [], widths[0]
        for s, w in enumerate(widths):
            for j in range(n):
                stride = 2 if (s > 0 and j == 0) else 1
                blocks.append(BasicBlock(cin, w, stride, residual))
                cin = w
        self.blocks = nn.Sequential(*blocks)
        self.fc = nn.Linear(widths[-1], num_classes)
        for m in self.modules():          # Kaiming (He) initialisation
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        x = F.relu(self.bn(self.conv(x)))
        x = self.blocks(x)
        x = F.adaptive_avg_pool2d(x, 1).flatten(1)
        return self.fc(x)
```

In [ ]:
display(pd.read_csv(os.path.join(R, 'task3b/table.csv')))

In [ ]:
display(Image(filename=os.path.join(R, 'task3b/plain_vs_residual.png')))

#### Supplementary: depth 56 at lr = 0.01\nPlain-56 diverged at lr = 0.1, so both depth-56 networks were rerun at lr = 0.01 (otherwise identical): `python task3b_residual.py --depths 56 --lr 0.01`.

In [ ]:
display(pd.read_csv(os.path.join(R, 'task3b_lr001/table.csv')))

In [ ]:
display(Image(filename=os.path.join(R, 'task3b_lr001/plain_vs_residual.png')))

**Interpretation.**

Main run (lr = 0.1, 30 epochs, flip + crop augmentation; plain and residual nets have *identical* parameter counts because the option-A shortcut has no parameters):

| Depth | Plain: train loss / val acc | Residual: train loss / val acc |
|---|---|---|
| 8 | 0.410 / 84.2% | 0.435 / 83.6% |
| 20 | 0.289 / 86.9% | 0.303 / 86.9% |
| 56 | **NaN (diverged in epoch 1)** / 10.0% | 0.400 / 82.5% |

- **Depth 8 and 20:** plain and residual are essentially the same. Shallow networks with batch norm are easy to optimise with or without shortcuts, and going from 8 to 20 layers lowers the training loss for both.
- **Depth 56 at lr 0.1:** the plain network's loss became NaN in the first epoch, so training stopped and it stayed at chance (10%). The residual network with exactly the same weights' shapes, initialisation and learning rate trained normally. This is already a sign of how badly conditioned the deep plain network is: the gradient through 56 stacked layers is unstable enough that lr = 0.1 blows it up.

**Supplementary run (depth 56 only, lr = 0.01)**, added so that the plain-56 network can train without diverging (He et al. also used a smaller learning rate to start their deepest networks):

| Model | Train loss | Train acc | Val acc |
|---|---|---|---|
| plain-56 (lr 0.01) | **1.496** | 45.1% | 46.7% |
| resnet-56 (lr 0.01) | 0.338 | 88.3% | 84.0% |
| plain-20 (lr 0.1, main run) | 0.289 | 90.0% | 86.9% |

- **This is the degradation problem.** Plain-56 has a far *higher training loss* (1.50) than plain-20 (0.29), and it is still stuck at 1.50 after the learning rate decays. Because it is the *training* loss that is worse, this is not overfitting; the deeper plain network is simply harder to optimise, even though it could in principle copy plain-20 and set the extra 36 layers to the identity.
- At the same learning rate, **resnet-56 trains without difficulty** (train loss 0.34, val 84.0%), so the shortcut is the only difference that matters:
  - each block only needs to learn a residual F(x) on top of the identity, so a near-identity mapping is easy to represent (F ≈ 0), and
  - the identity path gives gradients a direct route to the early layers, so they do not have to pass through every convolution.
- One caveat: in our 30-epoch budget resnet-56 (82.5% at lr 0.1, 84.0% at lr 0.01) does not beat resnet-20 (86.9%). The shortcut makes the deep network *trainable*, but 853k parameters at depth 56 need a longer schedule than 30 epochs to pay off; in He et al. (160+ epochs) the deeper ResNets were better.

## Task 4: Best CNN (<= 500k parameters, <= 20 epochs)

```python
class Residual(nn.Module):
    def __init__(self, c, shortcut=True):
        super().__init__()
        self.f = nn.Sequential(conv_bn_relu(c, c), conv_bn_relu(c, c))
        self.shortcut = shortcut

    def forward(self, x):
        return x + self.f(x) if self.shortcut else self.f(x)


class Scale(nn.Module):
    def __init__(self, s):
        super().__init__()
        self.s = s

    def forward(self, x):
        return x * self.s


class Task4Net(nn.Module):
    """ResNet-9-style network scaled down to < 500k parameters."""
    def __init__(self, c=(32, 64, 96, 192), shortcut=True, num_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            conv_bn_relu(3, c[0]),
            conv_bn_relu(c[0], c[1]), nn.MaxPool2d(2), Residual(c[1], shortcut),   # 16x16
            conv_bn_relu(c[1], c[2]), nn.MaxPool2d(2), Residual(c[2], shortcut),   # 8x8
            conv_bn_relu(c[2], c[3]), nn.MaxPool2d(2),                              # 4x4
            nn.AdaptiveMaxPool2d(1), nn.Flatten(),
            nn.Linear(c[3], num_classes, bias=False), Scale(0.125))

    def forward(self, x):
        return self.net(x)
```

```python
FULL = dict(shortcut=True, aug=["flip", "crop", "cutout"], cutout_size=8,
            opt="sgd", lr=0.1, wd=5e-4, label_smoothing=0.1, bs=256)
ABLATIONS = {                                  # each changes ONE design choice
    "full":              {},
    "no_shortcut":       dict(shortcut=False),
    "no_augmentation":   dict(aug=[]),
    "flip_crop_only":    dict(aug=["flip", "crop"]),
    "no_weight_decay":   dict(wd=0.0),
    "adamw_instead_sgd": dict(opt="adamw", lr=2e-3, wd=5e-2),
    "no_label_smoothing": dict(label_smoothing=0.0),
}
```

In [ ]:
display(pd.read_csv(os.path.join(R, 'task4/ablation_table.csv')))

In [ ]:
display(Image(filename=os.path.join(R, 'task4/ablation.png')))

In [ ]:
print(json.load(open(os.path.join(R,'task4/TEST_EVALUATED.json'))))

**Interpretation.**

The final architecture is a scaled-down ResNet-9 (483,424 parameters, under the 500k limit) trained for 20 epochs with SGD + momentum, a one-cycle learning-rate schedule (peak 0.1), weight decay 5e-4, label smoothing 0.1 and flip + crop + cutout augmentation. Each run took about 35 s on the RTX 6000 Ada. Each ablation changes **one** choice relative to `full`:

| Run | Final val acc | Δ vs full |
|---|---|---|
| flip + crop only (no cutout) | **91.12%** | +0.20 |
| no label smoothing | 91.08% | +0.16 |
| full | 90.92% | – |
| no weight decay | 90.86% | −0.06 |
| no shortcut | 90.62% | −0.30 |
| AdamW instead of SGD | 90.56% | −0.36 |
| no augmentation | 87.44% | **−3.48** |

- **Augmentation is the only choice that clearly matters.** Without it the model reaches 100% train accuracy by epoch 16 and validation stops at 87.4%. It memorises, as in Task 3(a).
- **All other differences are within ±0.4 points.** With a single seed and a 5k validation set (1 image = 0.02%), differences of this size are within run-to-run noise, so they should not be over-interpreted:
  - Cutout on top of flip + crop did not help in a 20-epoch budget; the model is not overfitting enough to need it.
  - The shortcut has little effect because this network is shallow; Task 3(b) showed shortcuts matter for deep networks.
  - Label smoothing changes the training loss a lot (0.68 with smoothing vs 0.18 without, since a smoothed target cannot be fitted with zero loss) but barely changes accuracy.
  - Tuned AdamW and tuned SGD perform about the same.
- **Selection:** the configuration with the highest validation accuracy, **flip_crop_only**, was chosen using the validation set only. The test set was then evaluated **once** (the guard file `TEST_EVALUATED.json` prevents re-evaluation).

**Test result: 90.77% test accuracy** (test loss 0.348) with 483,424 parameters.
- This is 0.35 points below its validation accuracy (91.12%). A small drop is expected: choosing the best of 7 runs on validation slightly favours a run that was lucky on that particular split. The gap is small, so the validation set was a reliable stand-in for the test set.

## Use of AI tools

I used Claude (Anthropic's AI assistant) while working on this assignment: to help write and debug the code, set up the Python environment and run the experiments on the GPU server, and to draft parts of the interpretation text. I checked the code and every reported number against the saved results myself, and I understand how each part works.